# ECLIPSE — w/o EC, Seed 1: Reuse Training and Recover Selected TEST Receipts

**All three 500-epoch runs already exist. This notebook does not train.**
Native uncalibrated graph hashes match the exact delta-zero operators, including data/candidate/config checks.
Attach `eclipse_wo_ec_seed1_reuse_bundle.zip` as a Kaggle input, or set `REUSE_ROOT` to its extracted folder.
Attach the processed MMRec datasets. Run cells in order on a GPU with internet for pinned source checkouts.

Baby's selected checkpoint is **epoch 430**; its six-decimal summary at epoch 500 is not selected TEST.
All datasets are re-evaluated from the original best checkpoint with the original native FreeRec evaluator.
A new observer records existing AverageMeters before reset; metric/scorer/masking code is unchanged.
Historical logs supply curves at **four-decimal log precision**, not fabricated unrounded curves.

After recovery, use `eclipse_wo_ec_seed2.ipynb` for the next independent training seed.
Never count this reevaluation as a new seed or a new training attempt.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, sys, uuid, subprocess, shutil, zipfile, re, math, time

SEED = 1
SELECTED_DATASETS = ('sports', 'baby', 'electronics')
GPU_INDEX = 0
REUSE_ROOT = None  # Extracted folder containing reuse_audit.json and baby/sports/electronics folders.
REUSE_ZIP = None   # Optional explicit path to eclipse_wo_ec_seed1_reuse_bundle.zip.
INSTALL_MISSING_DEPENDENCIES = True
AUTO_DOWNLOAD_DATA = True
STRICT_RUNTIME = True
DATASET_SOURCE_OVERRIDES = {}
REPO_URL = 'https://github.com/ThanhChuong12/STAIR-Enhanced.git'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_ROOT = WORK_DIR / 'eclipse_runs' / f'eclipse_wo_ec_seed1_reuse_{SESSION_ID}'
for name in ('logs','artifacts','configs','figures','tables','provenance','helpers'):
    (RUN_ROOT / name).mkdir(parents=True, exist_ok=True)
DATA_ROOT = WORK_DIR / 'eclipse_data'
PRIMARY_METRICS = ('RECALL@10','RECALL@20','NDCG@10','NDCG@20')
DATASET_TITLES = {'baby':'Amazon Baby','sports':'Amazon Sports','electronics':'Amazon Electronics'}
DATASET_FOLDERS = {k:f'Amazon2014{v}_550_MMRec' for k,v in
                   [('baby','Baby'),('sports','Sports'),('electronics','Electronics')]}
if SEED != 1 or not SELECTED_DATASETS or len(set(SELECTED_DATASETS)) != len(SELECTED_DATASETS):
    raise ValueError('This notebook recovers exactly seed 1; select datasets without duplicates.')
if set(SELECTED_DATASETS) - DATASET_TITLES.keys():
    raise ValueError('Unknown dataset.')
if isinstance(GPU_INDEX, bool) or not isinstance(GPU_INDEX,int) or GPU_INDEX < 0:
    raise ValueError('Invalid GPU index.')
def file_sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()
def write_json(path,payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(payload,indent=2,allow_nan=False)+'\n',encoding='utf-8')
print('Evaluation only. No fit(), train(), optimizer.step(), or checkpoint rewriting.')
print(RUN_ROOT)


In [ ]:
SEED1_REFERENCE = json.loads('{"baby":{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"lr":0.001,"weight_decay":0.3,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb"},{"file":"visual_modality.pkl","sha256":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5"}],"knn":[{"features":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb","k":5,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2cb2ad16db4996b65a179bbb221bbf09ac3eb390cf2cacf2cb3b26fd471efa40","cache_hit":false,"candidate_sha256":"5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827"},{"features":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5","k":1,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"a50d5b1b5a586f38a4a5aec66d128556d43046d899d631a8756b476d47e04586","cache_hit":false,"candidate_sha256":"ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.3,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"splits":{"test.txt":"307162309a9fae52172dabbbb0f373e845af5e73740d987b70bb0f406d5ac411","train.txt":"bcced50dea427734d76b031f1de140f219d67ba6bb6bc5eac85d7b3be51645bb","valid.txt":"c44687959888d29e2d931aa3de1210f7f708e204d169b7680e5adfcf0a183a3a"},"candidate_hashes":["5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827","ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"de38b811bc05e4838c2f52d025ee3464447a1af4e1a1844747635cddafd13a9e","graph_fingerprint":"b8013795d439064965dcd6946a44751d9a275ca0cefdc5231192066582d757a8"},"selected_receipt":{"mode":"test","epoch":480,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":1,"evidence_mode":"pooled_activity_excess","evidence_delta":1.0,"graph_fingerprint":"b8013795d439064965dcd6946a44751d9a275ca0cefdc5231192066582d757a8","metrics":{"RECALL@10":0.06861877473361941,"RECALL@20":0.10610715503188693,"NDCG@10":0.03679847170953948,"NDCG@20":0.046397878786352126}},"provenance":"logs/GD6/STAIR-PEEC/stair_icaart/baby_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","fit_seconds":1214.194699},"sports":{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"lr":0.001,"weight_decay":0.1,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2"},{"file":"visual_modality.pkl","sha256":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959"}],"knn":[{"features":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2","k":5,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9dacd55f8d2be69d2bebff457c2c83345135e05f90eb44c1c4f0b8d62b4347e7","cache_hit":false,"candidate_sha256":"bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af"},{"features":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959","k":1,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9a875c92e35fbe67a4e43ac1298d898fed91f7160d449e4d9e8d5b86899889e5","cache_hit":false,"candidate_sha256":"e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"splits":{"test.txt":"218501613d897422d24d2035e268af027f4ca77688a3f836080b197621f5610a","train.txt":"2491704e8543dbdeb9046cec0928c6734aa80521b1026afd53e0d296f4f7427c","valid.txt":"6dabc7efcc14de50909c6b1995f7bf9862d6233a3440d593d389e1bf91fef56e"},"candidate_hashes":["bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af","e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"383929cee0a814ae8211bde6c15ef0abb6ecd4499822bdf5f3579cb87fa04643","graph_fingerprint":"6b7be789b969cd17a17459deea25bfdf5bfcf518a008b02b7c2925a2d3f88f6c"},"selected_receipt":{"mode":"test","epoch":500,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":1,"evidence_mode":"pooled_activity_excess","evidence_delta":1.0,"graph_fingerprint":"6b7be789b969cd17a17459deea25bfdf5bfcf518a008b02b7c2925a2d3f88f6c","metrics":{"RECALL@10":0.07624274717468152,"RECALL@20":0.11450039259944982,"NDCG@10":0.04218513022410655,"NDCG@20":0.05208227600724625}},"provenance":"logs/GD6/STAIR-PEEC/stair_icaart/sports_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","fit_seconds":2520.869279},"electronics":{"commit":"5212263e87c5c1cfe6e48c91afa90201a48f5929","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"lr":0.001,"weight_decay":0.1,"batch_size":4096,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"cl_chunk_size":128,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0},"icaart_config":{"evidence_mode":"pooled_activity_excess","evidence_activity_bins":"log2","evidence_halves":0,"evidence_seed":null,"evidence_tau":2.0,"evidence_delta":1.0,"evidence_candidate_block":8192,"evidence_placebo_seed":20261016},"features":[{"file":"textual_modality.pkl","sha256":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e"},{"file":"visual_modality.pkl","sha256":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17"}],"knn":[{"features":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e","k":5,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2459d9b74920fa59d566d62171cbc1434d1c5e98d263d71c6561c2e7b2c9f695","cache_hit":false,"candidate_sha256":"a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54"},{"features":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17","k":1,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"6488fd7aaee647ded9774efd93bfe5c61ee399d4c36a23505569352cb8411b75","cache_hit":false,"candidate_sha256":"536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"}],"contract":{"config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":4096,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":128,"metadata_fingerprint":null},"splits":{"test.txt":"a5f4a1c7eb973d5d6bac5e6f023fcdedf1a66a46805714d233b4eb1fdb468234","train.txt":"495c58aec6c0b23c6780710d58e4718504af26175da8e01855b1f22675062e63","valid.txt":"145ce0ab06bf85260b74ba50829ab5041a2fc5c867d542c81e42da8aefefb51a"},"candidate_hashes":["a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54","536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"],"sources":{"stair5_v8.py":"dea612a122c82f7cc6c4c1ba069510fa1242fa3819f00b6cbfe0d37b9ed9c502","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"de46f3dd9bc11bb69acd5f151d175968e73d980be37ad36d5afc636b0b9c5e90","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8","stair_icaart.py":"77b862894f9aa91eba8fa96ad54a38eb6e29a25d9e134b7718a58f4bb5851014","stair_icaart_evidence.py":"11487e0fef87077cde7863b58847bcf22adeebef6e5378ae32f71f366d88c41a","stair_icaart_graph.py":"33bd582c8cd3b298eed1ce47ad99837ec9f0cd220778d684ae3d96161a455aad","stair_icaart_config.py":"fe5bcaa711ba64fd40d431db27ae6278e7c937ce05f57786b6f7262668061a51","main_stair_icaart.py":"b95eb5423ac8947da290bd13562e0aeb433adb8742ca0f0246139b4880aeec3f","stair_icaart_reporting.py":"6902360e9ac92dd511ea87e988ad622cae51886eb0ac183b170ce7df1b5065cc"},"data_fingerprint":"84680e889deeecb904f4475620eb0d638da332a95202c193c73c68924b0f0a82","graph_fingerprint":"43b9d0d255d7c7e72322fbeb834b0bc08206fcb236680db373aa30b6b6040242"},"selected_receipt":{"mode":"test","epoch":485,"step":-1,"selected_checkpoint":true,"selection":"validation NDCG@20","seed":1,"evidence_mode":"pooled_activity_excess","evidence_delta":1.0,"graph_fingerprint":"43b9d0d255d7c7e72322fbeb834b0bc08206fcb236680db373aa30b6b6040242","metrics":{"RECALL@10":0.04640053988342894,"RECALL@20":0.06811273359579144,"NDCG@10":0.026213868650622203,"NDCG@20":0.03183061497051809}},"provenance":"logs/GD6/STAIR-PEEC/electronics/stair_icaart/electronics_STAIR_ICAART_pooled_activity_excess_d1.0_seed1","fit_seconds":31999.033542}}')
EXPECTED_AUDIT = json.loads('{"baby":{"dataset":"baby","arm":"wo_ec","seed":1,"status":"REUSED_TRAINING_PENDING_SELECTED_REEVALUATION","source_directory":"logs/GD5/stair5_v8_artifacts/stair5_v8/baby_WMSG-core_seed1","source_log":"logs/GD5/stair5_v8_artifacts/stair5_v8/baby_WMSG-core_seed1.log","training_commit":"014886e7e7cd03096c724ba67815907e87e05d7c","runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4"},"selected_epoch":430,"historical_selected_test_rounded_4dp":{"RECALL@1":0.0128,"RECALL@10":0.0675,"RECALL@20":0.1054,"NDCG@10":0.0365,"NDCG@20":0.0462},"historical_fit_seconds":1158.378381,"checks":{"splits":true,"features":true,"candidate_hashes":true,"data_fingerprint":true,"v8_hyperparameters":true,"common_checkpoint_configuration":true,"native_delta_zero_operator":true,"seed_selection_precision":true,"recorded_sources_verified_at_recorded_commit":true,"all_500_epochs":true,"normal_fit_completion":true},"files":{"manifest.json":{"sha256":"31c04e86e45bfb9afd2c23883608447b4d97c63adf5f2ce88d0ea3ff4d2862c3","bytes":9983},"best_model.pt":{"sha256":"fa19032885f03e28efabcf9e867d15ad48cb1fdfdd2144d7108219e9136e6105","bytes":10914316},"training_telemetry.jsonl":{"sha256":"e338526062e19138a0d97451cb982d36d93a69d44192bf570de5b5cee3d889dd","bytes":178722},"training.log":{"sha256":"aa46fd72f3458f2dccc445d40ce2dd01d7f6d253e50189e06b3fb04a2348ad17","bytes":129058}},"source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","legacy_manifest":{"argv":["/kaggle/working/STAIR-Enhanced/main_stair5_v8.py","--config","/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_STAIR5_v8.yaml","--root","/kaggle/data","--device","0","--ranking","full","--artifact-dir","/kaggle/working/runs/stair5_v8/20261009_061008_a91d14/baby_WMSG-core_seed1","--v8-arm","WMSG-core","--semantic-mode","weighted","--edge-power","2.0","--edge-floor","0.05","--semantic-mix","1.0","--alignment","off","--anchor-seed","2718","--anchor-per-stratum","1024","--relation-gate","off","--relation-strength","0.0","--relation-threshold","0.7","--relation-temperature","0.1","--bsc-residual","0.0","--lambda-dirichlet","0.0","--seed","1","--eval-freq","5","--eta","0.1","--k-cf","5","--c-min","2","--t-shrinkage","5.0","--cf-block-size","64","--cf-memory-budget-mib","128","--lambda-nlgcl","0.01","--nlgcl-tau","0.2","--nlgcl-G","1","--nlgcl-alpha","0.5","--cl-chunk-size","1024","--knn-chunk-size","1024","--knn-device","cpu","--graph-cache-dir","/kaggle/working/runs/stair5_v8/graph_cache"],"commit":"014886e7e7cd03096c724ba67815907e87e05d7c","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","device":"cuda:0","cuda_available":true,"gpu":"Tesla T4","arm":"WMSG-core","seed":1,"graph":{"version":8,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"relation_gate":"off","relation_strength":0.0,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"num_users":19445,"num_items":7050,"cache_hit":false,"modality_0_edges_count":35250,"modality_0_cosine_mean":0.733953595161438,"modality_0_cosine_std":0.11293145269155502,"modality_0_nonpositive_cosine_fraction":0.0,"modality_0_weight_mean":0.5738692879676819,"modality_0_weight_min":0.13981308043003082,"modality_0_weight_max":1.0,"modality_1_edges_count":7050,"modality_1_cosine_mean":0.6781584024429321,"modality_1_cosine_std":0.12048749625682831,"modality_1_nonpositive_cosine_fraction":0.0,"modality_1_weight_mean":0.5006951689720154,"modality_1_weight_min":0.17312704026699066,"modality_1_weight_max":1.0,"wmsg_symmetric_nnz":59852,"wmsg_max_weight":2.0,"wmsg_min_weight":0.13981308043003082,"rpg_active":false,"active":true,"effective_eta":0.1,"s8_nnz":84138,"s0_nnz":59852,"cf_nnz":23054,"relative_operator_delta_vs_s0":0.2833131945359231,"graph_build_seconds":0.4915321499993297,"graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041"},"features":[{"file":"textual_modality.pkl","sha256":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb"},{"file":"visual_modality.pkl","sha256":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5"}],"knn":[{"features":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb","k":5,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2cb2ad16db4996b65a179bbb221bbf09ac3eb390cf2cacf2cb3b26fd471efa40","cache_hit":false,"candidate_sha256":"5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827"},{"features":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5","k":1,"chunk":1024,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"a50d5b1b5a586f38a4a5aec66d128556d43046d899d631a8756b476d47e04586","cache_hit":false,"candidate_sha256":"ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"}],"sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":7050,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"checkpoint_contract":{"version":8,"arm":"WMSG-core","lambda_nlgcl":0.01,"config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.3,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"data_fingerprint":"de38b811bc05e4838c2f52d025ee3464447a1af4e1a1844747635cddafd13a9e","graph_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":7050,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"sap_transform_sha256":null,"candidate_hashes":["5bfeae7c9bb0a96be4ca20803e20d0d84a57068aeb322749bd132f0ce2691827","ce97e02fbb82f94038c40fd2ae8e911c387c951b7710ee29eb2b144fc3cd7b08"],"features":[{"file":"textual_modality.pkl","sha256":"ceda3582c04f4e276e8d4bdcc2146f2634f6ca171e1f9dfafeb219dc5cfe96eb"},{"file":"visual_modality.pkl","sha256":"763bf9510c74fd5c6a45ee8164ca9f1ababc5c355c1757fc3acbdcc5d46af0a5"}],"splits":{"test.txt":"307162309a9fae52172dabbbb0f373e845af5e73740d987b70bb0f406d5ac411","train.txt":"bcced50dea427734d76b031f1de140f219d67ba6bb6bc5eac85d7b3be51645bb","valid.txt":"c44687959888d29e2d931aa3de1210f7f708e204d169b7680e5adfcf0a183a3a"},"sources":{"stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"torch":"2.11.0+cu128","freerec":"0.9.7"},"effective_lambda_nlgcl":0.01,"precision":{"tf32":false,"amp":false},"preprocessing_seconds":24.106930313999328,"preprocessing_gpu_memory":{"peak_allocated_bytes":512,"peak_reserved_bytes":2097152},"metadata_fingerprint":null,"ranking":"full","selection":"validation NDCG@20","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.1,"lr":0.001,"weight_decay":0.3,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0,"v4_support_files":[],"metadata_file":"","placebo_seed":1},"freerec_log_path":"./logs/STAIR5-v8-WMSG-CSE/Amazon2014Baby_550_MMRec/1009065247","test_records":"Native FreeRec final and selected; report selected only."},"full_reference_graph_fingerprint":"b8013795d439064965dcd6946a44751d9a275ca0cefdc5231192066582d757a8","full_reference_baseline_v8_fingerprint":"eab753510c9d830425ba0f1f44f6d4964aa0143c574d4707fb2e7c9d93532041","precision_caveat":"Historical selected TEST line has four decimals. Do not treat summary-at-500 as selected-at-430 for Baby.","evaluation_receipt_status":"NOT_MEASURED_BY_THIS_AUDIT"},"sports":{"dataset":"sports","arm":"wo_ec","seed":1,"status":"REUSED_TRAINING_PENDING_SELECTED_REEVALUATION","source_directory":"logs/GD5/stair5_v8_artifacts/stair5_v8/sports_WMSG-core_seed1","source_log":"logs/GD5/stair5_v8_artifacts/stair5_v8/sports_WMSG-core_seed1.log","training_commit":"014886e7e7cd03096c724ba67815907e87e05d7c","runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4"},"selected_epoch":500,"historical_selected_test_rounded_4dp":{"RECALL@1":0.0151,"RECALL@10":0.0762,"RECALL@20":0.1142,"NDCG@10":0.042,"NDCG@20":0.0518},"historical_fit_seconds":2499.692581,"checks":{"splits":true,"features":true,"candidate_hashes":true,"data_fingerprint":true,"v8_hyperparameters":true,"common_checkpoint_configuration":true,"native_delta_zero_operator":true,"seed_selection_precision":true,"recorded_sources_verified_at_recorded_commit":true,"all_500_epochs":true,"normal_fit_completion":true},"files":{"manifest.json":{"sha256":"5d54a92ff79e1b8c00763442105aabd2a0d0eec46eb271bf0d9c5a5a80dc1d26","bytes":10864},"best_model.pt":{"sha256":"f1beede0258de52124b56130f8611f9f32299e571f60789d070808356858d4a1","bytes":22116684},"training_telemetry.jsonl":{"sha256":"08784453e4142815dcf0028d0cb6bd271c7cbd1bf60fffc9355131a6983feaed","bytes":179769},"training.log":{"sha256":"a02f38e5928654f897478e9ae3b9424aeafc638414554677571ca7f0eec74585","bytes":129338}},"source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","legacy_manifest":{"argv":["/kaggle/working/STAIR-Enhanced/main_stair5_v8.py","--config","/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_STAIR5_v8.yaml","--root","/kaggle/data","--device","0","--ranking","full","--artifact-dir","/kaggle/working/runs/stair5_v8/20261009_061008_a91d14/sports_WMSG-core_seed1","--v8-arm","WMSG-core","--semantic-mode","weighted","--edge-power","2.0","--edge-floor","0.05","--semantic-mix","1.0","--alignment","off","--anchor-seed","2718","--anchor-per-stratum","1024","--relation-gate","off","--relation-strength","0.0","--relation-threshold","0.7","--relation-temperature","0.1","--bsc-residual","0.0","--lambda-dirichlet","0.0","--seed","1","--eval-freq","5","--eta","0.1","--k-cf","5","--c-min","2","--t-shrinkage","5.0","--cf-block-size","64","--cf-memory-budget-mib","128","--lambda-nlgcl","0.01","--nlgcl-tau","0.2","--nlgcl-G","1","--nlgcl-alpha","0.5","--cl-chunk-size","1024","--knn-chunk-size","1024","--knn-device","cpu","--graph-cache-dir","/kaggle/working/runs/stair5_v8/graph_cache"],"commit":"014886e7e7cd03096c724ba67815907e87e05d7c","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","device":"cuda:0","cuda_available":true,"gpu":"Tesla T4","arm":"WMSG-core","seed":1,"graph":{"version":8,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"relation_gate":"off","relation_strength":0.0,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"num_users":35598,"num_items":18357,"cache_hit":false,"modality_0_edges_count":91785,"modality_0_cosine_mean":0.7440291047096252,"modality_0_cosine_std":0.11236167699098587,"modality_0_nonpositive_cosine_fraction":0.0,"modality_0_weight_mean":0.5878943204879761,"modality_0_weight_min":0.1342153549194336,"modality_0_weight_max":1.0,"modality_1_edges_count":18357,"modality_1_cosine_mean":0.7225796580314636,"modality_1_cosine_std":0.13192041218280792,"modality_1_nonpositive_cosine_fraction":0.0,"modality_1_weight_mean":0.562548041343689,"modality_1_weight_min":0.15613286197185516,"modality_1_weight_max":1.0,"wmsg_symmetric_nnz":157744,"wmsg_max_weight":2.0,"wmsg_min_weight":0.1342153549194336,"rpg_active":false,"cf_cache_hit":true,"active":true,"effective_eta":0.1,"s8_nnz":206296,"s0_nnz":157744,"cf_nnz":43334,"relative_operator_delta_vs_s0":0.2968943678218056,"graph_build_seconds":0.5968075470000258,"graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea"},"features":[{"file":"textual_modality.pkl","sha256":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2"},{"file":"visual_modality.pkl","sha256":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959"}],"knn":[{"features":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2","k":5,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9dacd55f8d2be69d2bebff457c2c83345135e05f90eb44c1c4f0b8d62b4347e7","cache_hit":true,"loaded_signature":{"features":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2","k":5,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038"},"candidate_sha256":"bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af","frozen_support":false},{"features":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959","k":1,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"9a875c92e35fbe67a4e43ac1298d898fed91f7160d449e4d9e8d5b86899889e5","cache_hit":true,"loaded_signature":{"features":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959","k":1,"chunk":609,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038"},"candidate_sha256":"e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5","frozen_support":false}],"sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":18357,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"checkpoint_contract":{"version":8,"arm":"WMSG-core","lambda_nlgcl":0.01,"config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":1024,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":1024,"metadata_fingerprint":null},"data_fingerprint":"383929cee0a814ae8211bde6c15ef0abb6ecd4499822bdf5f3579cb87fa04643","graph_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":18357,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"sap_transform_sha256":null,"candidate_hashes":["bbe9e918209a03b5186a74cbd78ba474f420dfaab4eb6c2095c7c578ac0ca5af","e3270692a18ae7d11d50a83d30b7f071f4a095e630714c01b2a0f6b4cff4e8a5"],"features":[{"file":"textual_modality.pkl","sha256":"ca99972f3a2c7a9ead14e9a0d649351411722bbd98e920b4c5fe9f8ffa1564a2"},{"file":"visual_modality.pkl","sha256":"5d2ea8187bbc52f806616404815930061acefe6cb22f353e3f02f286c95b9959"}],"splits":{"test.txt":"218501613d897422d24d2035e268af027f4ca77688a3f836080b197621f5610a","train.txt":"2491704e8543dbdeb9046cec0928c6734aa80521b1026afd53e0d296f4f7427c","valid.txt":"6dabc7efcc14de50909c6b1995f7bf9862d6233a3440d593d389e1bf91fef56e"},"sources":{"stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"torch":"2.11.0+cu128","freerec":"0.9.7"},"effective_lambda_nlgcl":0.01,"precision":{"tf32":false,"amp":false},"preprocessing_seconds":28.933243898,"preprocessing_gpu_memory":{"peak_allocated_bytes":512,"peak_reserved_bytes":2097152},"metadata_fingerprint":null,"ranking":"full","selection":"validation NDCG@20","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.2,"lr":0.001,"weight_decay":0.1,"batch_size":1024,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"cl_chunk_size":1024,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0,"v4_support_files":[],"metadata_file":"","placebo_seed":1},"freerec_log_path":"./logs/STAIR5-v8-WMSG-CSE/Amazon2014Sports_550_MMRec/1009061020","test_records":"Native FreeRec final and selected; report selected only."},"full_reference_graph_fingerprint":"6b7be789b969cd17a17459deea25bfdf5bfcf518a008b02b7c2925a2d3f88f6c","full_reference_baseline_v8_fingerprint":"cc0ff8deb6bbdac1c9295e27781dbb96de422731779794d79b4e1fd355ee75ea","precision_caveat":"Historical selected TEST line has four decimals. Do not treat summary-at-500 as selected-at-430 for Baby.","evaluation_receipt_status":"NOT_MEASURED_BY_THIS_AUDIT"},"electronics":{"dataset":"electronics","arm":"wo_ec","seed":1,"status":"REUSED_TRAINING_PENDING_SELECTED_REEVALUATION","source_directory":"logs/GD5/stair5_v8_artifacts/electronics_WMSG/stair5_v8/electronics_WMSG-core_seed1","source_log":"logs/GD5/stair5_v8_artifacts/electronics_WMSG/stair5_v8/electronics_WMSG-core_seed1.log","training_commit":"d401a27511a3f77b13c082d68c7f2359b7e4e61c","runtime":{"python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","gpu":"Tesla T4"},"selected_epoch":500,"historical_selected_test_rounded_4dp":{"RECALL@1":0.0102,"RECALL@10":0.046,"RECALL@20":0.068,"NDCG@10":0.026,"NDCG@20":0.0317},"historical_fit_seconds":31602.465461,"checks":{"splits":true,"features":true,"candidate_hashes":true,"data_fingerprint":true,"v8_hyperparameters":true,"common_checkpoint_configuration":true,"native_delta_zero_operator":true,"seed_selection_precision":true,"recorded_sources_verified_at_recorded_commit":true,"all_500_epochs":true,"normal_fit_completion":true},"files":{"manifest.json":{"sha256":"02aa9fc747d574c4dba723bc27e2b08a076b60df755df7f7689593a5bab5be2d","bytes":10004},"best_model.pt":{"sha256":"b268f27b5c869e6cbe2683a710a9cf9bfc34df59be4c8c685f96b549cb7d706d","bytes":106918220},"training_telemetry.jsonl":{"sha256":"39c4cb5b4b37c79d942ea94402e95271270a8ad3cac1586d20d0e35b376da79c","bytes":179699},"training.log":{"sha256":"3627c726eca21ae25d70a39abeffa87e4017c2ce6d06242ff665e6b5cf13a9e9","bytes":130930}},"source_hashes":{"models/stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","models/stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","models/stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","models/stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","models/stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","models/stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","models/stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","models/freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","optimizers/stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","optimizers/AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","legacy_manifest":{"argv":["/kaggle/working/STAIR-Enhanced/main_stair5_v8.py","--config","/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_STAIR5_v8.yaml","--root","/kaggle/data","--device","0","--ranking","full","--artifact-dir","/kaggle/working/runs/stair5_v8/20261009_074535_186774/electronics_WMSG-core_seed1","--v8-arm","WMSG-core","--semantic-mode","weighted","--edge-power","2.0","--edge-floor","0.05","--semantic-mix","1.0","--alignment","off","--anchor-seed","2718","--anchor-per-stratum","1024","--relation-gate","off","--relation-strength","0.0","--relation-threshold","0.7","--relation-temperature","0.1","--bsc-residual","0.0","--lambda-dirichlet","0.0","--seed","1","--eval-freq","5","--eta","0.1","--k-cf","5","--c-min","2","--t-shrinkage","5.0","--cf-block-size","64","--cf-memory-budget-mib","128","--lambda-nlgcl","0.01","--nlgcl-tau","0.2","--nlgcl-G","1","--nlgcl-alpha","0.5","--cl-chunk-size","128","--knn-chunk-size","1024","--knn-device","cpu","--graph-cache-dir","/kaggle/working/runs/stair5_v8/graph_cache"],"commit":"d401a27511a3f77b13c082d68c7f2359b7e4e61c","python":"3.13.15","torch":"2.11.0+cu128","freerec":"0.9.7","device":"cuda:0","cuda_available":true,"gpu":"Tesla T4","arm":"WMSG-core","seed":1,"graph":{"version":8,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"relation_gate":"off","relation_strength":0.0,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"num_users":192403,"num_items":63001,"cache_hit":false,"modality_0_edges_count":315005,"modality_0_cosine_mean":0.7866738438606262,"modality_0_cosine_std":0.1035960391163826,"modality_0_nonpositive_cosine_fraction":0.0,"modality_0_weight_mean":0.6481084227561951,"modality_0_weight_min":0.11206944286823273,"modality_0_weight_max":1.0,"modality_1_edges_count":63001,"modality_1_cosine_mean":0.7869994640350342,"modality_1_cosine_std":0.12887389957904816,"modality_1_nonpositive_cosine_fraction":0.0,"modality_1_weight_mean":0.6541777849197388,"modality_1_weight_min":0.16854384541511536,"modality_1_weight_max":1.0,"wmsg_symmetric_nnz":542712,"wmsg_max_weight":2.0,"wmsg_min_weight":0.11206944286823273,"rpg_active":false,"active":true,"effective_eta":0.1,"s8_nnz":739381,"s0_nnz":542712,"cf_nnz":174488,"relative_operator_delta_vs_s0":0.28176892036927004,"graph_build_seconds":4.608284074999915,"graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b"},"features":[{"file":"textual_modality.pkl","sha256":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e"},{"file":"visual_modality.pkl","sha256":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17"}],"knn":[{"features":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e","k":5,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"2459d9b74920fa59d566d62171cbc1434d1c5e98d263d71c6561c2e7b2c9f695","cache_hit":false,"candidate_sha256":"a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54"},{"features":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17","k":1,"chunk":177,"algorithm":"cosine_fp32_diagonal_minus10_torch_topk_v1","torch":"2.11.0+cu128","device":"cpu","tf32":false,"source":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","fingerprint":"6488fd7aaee647ded9774efd93bfe5c61ee399d4c36a23505569352cb8411b75","cache_hit":false,"candidate_sha256":"536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"}],"sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":63001,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"checkpoint_contract":{"version":8,"arm":"WMSG-core","lambda_nlgcl":0.01,"config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"num_neighbors":[5,1],"mfiles":["textual_modality.pkl","visual_modality.pkl"],"lr":0.001,"weight_decay":0.1,"beta1":0.9,"beta2":0.999,"batch_size":4096,"seed":1,"eval_freq":5,"ranking":"full","which4best":"NDCG@20","num_workers":4,"nlgcl_G":1,"nlgcl_tau":0.2,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"placebo_seed":1,"knn_device":"cpu","knn_chunk_size":1024,"cl_chunk_size":128,"metadata_fingerprint":null},"data_fingerprint":"84680e889deeecb904f4475620eb0d638da332a95202c193c73c68924b0f0a82","graph_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","sap_audit":{"alignment_mode":"off","status":"BYPASS_V4_PARITY","passed":true,"fallback_reason":null,"num_items":63001,"embedding_dim":64,"fused_formula":"V^{(0)} = (5*Z_t + Z_v) / 6"},"sap_transform_sha256":null,"candidate_hashes":["a7be45316c97702a72dcbce8d1595b2dca971af0a132611646ae430522b09d54","536c50370c58b370550c0e947d706b0fc3cadbf32d9f243f2ab0fbc5d0f744eb"],"features":[{"file":"textual_modality.pkl","sha256":"14a936f18890fbd8b4e6436061b971de878ba79005a2477325d3466e305d635e"},{"file":"visual_modality.pkl","sha256":"13c40105ff6872616e60d0418faa5c4eb4e14e0de1af7d1e71e6820df63dea17"}],"splits":{"test.txt":"a5f4a1c7eb973d5d6bac5e6f023fcdedf1a66a46805714d233b4eb1fdb468234","train.txt":"495c58aec6c0b23c6780710d58e4718504af26175da8e01855b1f22675062e63","valid.txt":"145ce0ab06bf85260b74ba50829ab5041a2fc5c867d542c81e42da8aefefb51a"},"sources":{"stair5_v8.py":"c022aeb43c00112df5d93a14724afbc239b5cfee25fc4d9bd8e714301975ae92","stair5_v8_alignment.py":"8d6bb9eb25dfe2426d2a358ec82268d27c9b4913eef0472b6fda83fbd231607d","stair5_v8_graph.py":"2a7c931740ceeac177ce3fc6dc2cdae479eb75673a8ef1b115122ba4a0826ae7","stair5_v4_graph.py":"eb5d2b2355871711931d597bbe2d01f520b174852430e922be268826c7252241","stair5_v4.py":"640f26b80b5b5eabf9bdcd44bac7a9c0544571168b31eefb7cbf96e59c166038","stair5_v4_objectives.py":"1afd078a22aad8635778cfa61f05628beaecd180cd2c6c17ed56f72a6a94182e","stair5_v4_utils.py":"39930cb54af3f5cdb65983f05b3b03cabe502673b9d70889b0ac7c1d22274b1a","freerec_compat.py":"708364b1874ab9c1a4d9ce0a21a2e31e57b9bfeac1da4c570af43a28bb80e626","stair5_v8_smoother.py":"c4f1ebab6008ffa5f7720007a17854040f0a7211fbd6087fc9dedd26d10d7663","AdamW.py":"43108b6264cd5b1fa723d0a731faff0269d05fb91bcf525445c9bd0406443715","main_stair5_v8.py":"036b45594a8c0b17e32c5401c5b84a93e85992fadae6303aa0db9eeaea8331b8"},"torch":"2.11.0+cu128","freerec":"0.9.7"},"effective_lambda_nlgcl":0.01,"precision":{"tf32":false,"amp":false},"preprocessing_seconds":280.474905628,"preprocessing_gpu_memory":{"peak_allocated_bytes":512,"peak_reserved_bytes":2097152},"metadata_fingerprint":null,"ranking":"full","selection":"validation NDCG@20","v8_config":{"embedding_dim":64,"num_layers":3,"gamma":0.4,"lr":0.001,"weight_decay":0.1,"batch_size":4096,"epochs":500,"beta1":0.9,"beta2":0.999,"mfiles":["textual_modality.pkl","visual_modality.pkl"],"num_neighbors":[5,1],"eval_freq":5,"lambda_nlgcl":0.01,"nlgcl_tau":0.2,"nlgcl_G":1,"nlgcl_alpha":0.5,"eta":0.1,"k_cf":5,"c_min":2,"t_shrinkage":5.0,"v8_arm":"WMSG-core","semantic_mode":"weighted","edge_power":2.0,"edge_floor":0.05,"semantic_mix":1.0,"alignment":"off","anchor_seed":2718,"anchor_per_stratum":1024,"relation_gate":"off","relation_strength":0.0,"relation_threshold":0.7,"relation_temperature":0.1,"bsc_residual":0.0,"lambda_dirichlet":0.0,"cl_chunk_size":128,"knn_chunk_size":1024,"knn_device":"cpu","cf_block_size":64,"cf_memory_budget_mib":128.0,"v4_support_files":[],"metadata_file":"","placebo_seed":1},"freerec_log_path":"./logs/STAIR5-v8-WMSG-CSE/Amazon2014Electronics_550_MMRec/1009074547","test_records":"Native FreeRec final and selected; report selected only."},"full_reference_graph_fingerprint":"43b9d0d255d7c7e72322fbeb834b0bc08206fcb236680db373aa30b6b6040242","full_reference_baseline_v8_fingerprint":"3de143b8c1d3b5fe5c8107964b1131be71d9b49b2eec36a37d6c160d08bcc22b","precision_caveat":"Historical selected TEST line has four decimals. Do not treat summary-at-500 as selected-at-430 for Baby.","evaluation_receipt_status":"NOT_MEASURED_BY_THIS_AUDIT"}}')
EXPECTED_BUNDLE_SHA256 = '052a00da2529a2b855ac5f6913619b7db448920f3f797e97906e4382e5b11be1'


## 1. Locate and verify the existing runs

An extracted bundle is accepted only if every original file hash matches the embedded audit.
A ZIP is checked before extraction and rejects unsafe paths. Original checkpoints remain read-only.
Missing artifacts stop this notebook; it will never silently launch a replacement training run.


In [ ]:
input_roots = [Path('/kaggle/input'), Path.cwd() / 'reports', Path.cwd()]
if REUSE_ROOT is None:
    candidates = []
    for root in input_roots:
        if root.is_dir():
            candidates += [p.parent for p in root.rglob('reuse_audit.json')
                           if all((p.parent/key/'best_model.pt').is_file() for key in SELECTED_DATASETS)]
    REUSE_ROOT = candidates[0] if candidates else None
if REUSE_ROOT is None:
    zip_paths = [Path(REUSE_ZIP)] if REUSE_ZIP else []
    if not zip_paths:
        for root in input_roots:
            if root.is_dir():
                zip_paths.extend(root.rglob('eclipse_wo_ec_seed1_reuse_bundle.zip'))
    if not zip_paths:
        raise FileNotFoundError('Attach eclipse_wo_ec_seed1_reuse_bundle.zip as a Kaggle input, or set REUSE_ROOT.')
    archive = zip_paths[0]
    if file_sha256(archive) != EXPECTED_BUNDLE_SHA256:
        raise ValueError(f'Unexpected reuse ZIP hash: {archive}')
    extract_to = RUN_ROOT / 'verified_reuse_input'
    extract_to.mkdir()
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            target = (extract_to/info.filename).resolve()
            if not target.is_relative_to(extract_to.resolve()) or info.file_size > 200*2**20:
                raise ValueError(f'Unsafe/unexpected ZIP entry: {info.filename}')
        z.extractall(extract_to)
    REUSE_ROOT = extract_to / 'eclipse_wo_ec_seed1_reuse'
REUSE_ROOT = Path(REUSE_ROOT).expanduser().resolve()
supplied = json.loads((REUSE_ROOT/'reuse_audit.json').read_text(encoding='utf-8'))
if supplied != EXPECTED_AUDIT:
    raise ValueError('Bundle audit differs from the verified attempt.')
for key in SELECTED_DATASETS:
    for name, info in EXPECTED_AUDIT[key]['files'].items():
        path = REUSE_ROOT/key/name
        if not path.is_file() or path.stat().st_size != info['bytes'] or file_sha256(path) != info['sha256']:
            raise ValueError(f'Original artifact hash mismatch: {path}')
    print(key, ': verified existing training; selected epoch', EXPECTED_AUDIT[key]['selected_epoch'])
write_json(RUN_ROOT/'provenance/reuse_audit.json',EXPECTED_AUDIT)


## 2. Original source checkouts and runtime

Separate detached checkouts preserve the checkpoint's strict source contract. No reset of a running checkout,
no `strict=False`, no edit to `_extra_state`. Python major/minor, Torch and FreeRec must match by default.
Torch stays in child processes; no notebook-kernel CUDA allocation.


In [ ]:
import importlib.metadata, importlib.util
if INSTALL_MISSING_DEPENDENCIES:
    packages = {
        'freerec': ('freerec==0.9.7', True),
        'torch_geometric': ('torch-geometric', True),
        'numpy': ('numpy', False), 'scipy': ('scipy', False),
        'yaml': ('PyYAML', False), 'pandas': ('pandas', False),
        'matplotlib': ('matplotlib', False), 'prettytable': ('prettytable', False),
        'sklearn': ('scikit-learn', False), 'psutil': ('psutil', False),
        'kagglehub': ('kagglehub', False),
    }
    for module, (requirement, no_deps) in packages.items():
        missing = importlib.util.find_spec(module) is None
        if module == 'freerec' and not missing:
            missing = importlib.metadata.version('freerec') != '0.9.7'
        if missing:
            command = [sys.executable, '-m', 'pip', 'install', '--quiet']
            if no_deps:
                command.append('--no-deps')
            subprocess.run(command + [requirement], check=True)

CHECKOUTS = {}
for key in SELECTED_DATASETS:
    rec = EXPECTED_AUDIT[key]
    checkout = WORK_DIR/'eclipse_legacy_sources'/rec['training_commit'][:12]
    checkout.parent.mkdir(parents=True,exist_ok=True)
    if not checkout.exists():
        subprocess.run(['git','clone','--no-checkout',REPO_URL,str(checkout)],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach',rec['training_commit']],check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip()
    if actual != rec['training_commit']:
        raise ValueError(f'Existing historical checkout is at another commit: {checkout}')
    for relative,digest in rec['source_hashes'].items():
        if file_sha256(checkout/relative) != digest:
            raise ValueError(f'Historical source bytes differ (including line endings): {checkout/relative}')
    CHECKOUTS[key] = checkout
REPO_ROOT = CHECKOUTS[SELECTED_DATASETS[0]]  # Data discovery only.
ENV = os.environ.copy()
ENV.update(PYTHONUTF8='1',PYTHONIOENCODING='utf-8',PYTHONHASHSEED='1',CUDA_VISIBLE_DEVICES=str(GPU_INDEX))
ENV.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
probe = "import json,platform,torch; import models.freerec_compat; import freerec; print('ECLIPSE_ENV='+json.dumps(dict(python=platform.python_version(),torch=str(torch.__version__),freerec=str(freerec.__version__),cuda=torch.cuda.is_available(),gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)))"
probe_env = dict(ENV,PYTHONPATH=str(REPO_ROOT))
check = subprocess.run([sys.executable,'-c',probe],cwd=REPO_ROOT,env=probe_env,
                       text=True,encoding='utf-8',capture_output=True)
if check.returncode:
    print(check.stdout,check.stderr); check.check_returncode()
RUNTIME_ENV = json.loads(next(line.split('=',1)[1] for line in check.stdout.splitlines() if line.startswith('ECLIPSE_ENV=')))
ref = EXPECTED_AUDIT[SELECTED_DATASETS[0]]['runtime']
deviations = {k:{'reference':ref[k],'current':RUNTIME_ENV[k]} for k in ('torch','freerec') if ref[k]!=RUNTIME_ENV[k]}
if RUNTIME_ENV['python'].split('.')[:2] != ref['python'].split('.')[:2]:
    deviations['python'] = {'reference':ref['python'],'current':RUNTIME_ENV['python']}
if not RUNTIME_ENV['cuda']:
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
write_json(RUN_ROOT/'provenance/environment.json',dict(runtime=RUNTIME_ENV,deviations=deviations))
if deviations and STRICT_RUNTIME:
    raise RuntimeError(f'Restore the historical runtime; deviations: {deviations}')
if deviations:
    print('PROTOCOL DEVIATION: do not silently pool this reevaluation with matched-runtime runs.',deviations)
freeze = subprocess.run([sys.executable,'-m','pip','freeze'],text=True,capture_output=True,check=True)
(RUN_ROOT/'provenance/pip_freeze.txt').write_text(freeze.stdout,encoding='utf-8')
print(RUNTIME_ENV)


## 3. Data fingerprints — identical splits and features, no preprocessing changes


In [ ]:
import shutil

REQUIRED_FILES = ('train.txt', 'valid.txt', 'test.txt', 'textual_modality.pkl', 'visual_modality.pkl')

def discover_datasets(extra_roots=()):
    found = {}
    roots = [REPO_ROOT / 'data', Path('/kaggle/data'), Path('/kaggle/input'), *map(Path, extra_roots)]
    aliases = {'baby': ('baby',), 'sports': ('sport',), 'electronics': ('electronic',)}
    visited = set()
    for root in roots:
        if not root.is_dir() or str(root.resolve()) in visited:
            continue
        visited.add(str(root.resolve()))
        for current, directories, files in os.walk(root, followlinks=False):
            directories[:] = [d for d in directories if d not in {'.git', 'chunks', '__pycache__', '.stair_icaart_graph_cache'}]
            if not set(REQUIRED_FILES).issubset(files):
                continue
            path = Path(current)
            for key in SELECTED_DATASETS:
                if any(alias in path.name.lower() for alias in aliases[key]):
                    found.setdefault(key, path)
    for key, path in DATASET_SOURCE_OVERRIDES.items():
        path = Path(path).expanduser()
        if not all((path / name).is_file() for name in REQUIRED_FILES):
            raise FileNotFoundError(f'Incomplete explicit dataset source: {path}')
        found[key] = path
    return found

sources = discover_datasets()
missing = set(SELECTED_DATASETS) - sources.keys()
if missing and AUTO_DOWNLOAD_DATA:
    import kagglehub
    downloaded = kagglehub.dataset_download('rainyle/stair-datasets-mmrec')
    sources.update(discover_datasets([downloaded]))
missing = set(SELECTED_DATASETS) - sources.keys()
if missing:
    raise FileNotFoundError(f'Attach the processed dataset or set DATASET_SOURCE_OVERRIDES for: {sorted(missing)}')

DATA_RECEIPTS = {}
for key in SELECTED_DATASETS:
    source = sources[key].resolve()
    reference = SEED1_REFERENCE[key]
    expected = dict(reference['contract']['splits'])
    expected.update({record['file']: record['sha256'] for record in reference['features']})
    destination = DATA_ROOT / 'Processed' / DATASET_FOLDERS[key]
    destination.mkdir(parents=True, exist_ok=True)
    for name, expected_hash in expected.items():
        original = source / name
        if not original.is_file():
            raise FileNotFoundError(original)
        actual_hash = file_sha256(original)
        if actual_hash != expected_hash:
            raise RuntimeError(f'Dataset differs from seed 1: {original}')
        target = destination / name
        if target.exists():
            if file_sha256(target) != expected_hash:
                raise RuntimeError(f'Existing linked/copied input has a different fingerprint: {target}')
        else:
            try:
                target.symlink_to(original)
            except OSError:
                shutil.copy2(original, target)
    (destination / 'chunks').mkdir(exist_ok=True)
    DATA_RECEIPTS[key] = {'source': str(source), 'processed': str(destination), 'fingerprints': expected}
    print(f'{DATASET_TITLES[key]}: split and feature fingerprints match seed 1.')
write_json(RUN_ROOT / 'provenance' / 'data_check.json', DATA_RECEIPTS)


## 4. Evaluation-only helper and immutable plans

The helper verifies the original model's complete extra-state contract, loads `best_model.pt` strictly,
and calls native `coach.test(selected_epoch)`. It refuses training. Metrics are captured before
native `step()` resets the AverageMeters. Recovered results must agree with historical selected log rounding.


In [ ]:
EVALUATION_HELPER = '"""Observe native FreeRec TEST meters for one original best checkpoint; no training."""\nimport hashlib, json, math, os, platform, sys, time\nfrom pathlib import Path\nimport models.freerec_compat\nimport freerec\nimport torch\nfrom main_stair5_v8 import build_config, load_dataset, CoachForSTAIR5_v8\nfrom models.stair5_v8 import STAIR5_v8_Model\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\ndef write(path, payload):\n    path = Path(path)\n    temporary = path.with_suffix(\'.json.tmp\')\n    temporary.write_text(json.dumps(payload,indent=2,allow_nan=False)+\'\\n\',encoding=\'utf-8\')\n    os.replace(temporary,path)\n\nif __name__ == \'__main__\':\n    plan = json.loads(Path(os.environ[\'ECLIPSE_REUSE_PLAN\']).read_text(encoding=\'utf-8\'))\n    target = Path(plan[\'output\'])\n    rec = plan[\'audit\']\n    original = rec[\'legacy_manifest\']\n    runtime = dict(python=platform.python_version(),torch=str(torch.__version__),freerec=str(freerec.__version__))\n    deviations = {k:dict(reference=rec[\'runtime\'][k],current=v) for k,v in runtime.items()\n                  if (v.split(\'.\')[:2] != rec[\'runtime\'][k].split(\'.\')[:2] if k==\'python\' else v != rec[\'runtime\'][k])}\n    if plan[\'strict_runtime\'] and deviations:\n        raise ValueError(f\'Child runtime mismatch: {deviations}\')\n    for relative,digest in rec[\'source_hashes\'].items():\n        if sha(Path.cwd()/relative) != digest:\n            raise ValueError(f\'Source digest mismatch: {relative}\')\n    checkpoint = Path(plan[\'checkpoint\'])\n    if sha(checkpoint) != rec[\'files\'][\'best_model.pt\'][\'sha256\']:\n        raise ValueError(\'Checkpoint digest mismatch.\')\n    cfg = build_config()  # Native parser + original CLI protocol and seeding.\n    expected_cfg = original[\'checkpoint_contract\'][\'config\']\n    for key, value in expected_cfg.items():\n        if getattr(cfg,key) != value:\n            raise ValueError(f\'Original protocol changed: {key}: {getattr(cfg,key)} != {value}\')\n    for key,value in original[\'v8_config\'].items():\n        if getattr(cfg,key) != value:\n            raise ValueError(f\'Original graph/loss configuration changed: {key}\')\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    dataset = load_dataset(cfg)\n    model = STAIR5_v8_Model(dataset,cfg)\n    if model.get_extra_state() != original[\'checkpoint_contract\']:\n        raise ValueError(\'Rebuilt model does not reproduce the exact checkpoint source/data/operator contract.\')\n    state = torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n    model.load_state_dict(state,strict=True)\n    del state\n    selected_epoch = rec[\'selected_epoch\']\n    class ReceiptCoach(CoachForSTAIR5_v8):\n        def fit(self):\n            raise RuntimeError(\'Training is forbidden in this recovery helper.\')\n        def train_per_epoch(self,epoch):\n            raise RuntimeError(\'Training is forbidden in this recovery helper.\')\n        @torch.no_grad()\n        def evaluate(self,epoch,step=-1,mode=\'valid\'):\n            if mode != \'test\' or epoch != selected_epoch:\n                raise ValueError(\'Only the preselected original TEST checkpoint may be evaluated.\')\n            result = super().evaluate(epoch,step,mode)\n            metrics = {}\n            for meters in self.monitors[mode].values():\n                for meter in meters:\n                    if meter.active and meter.count > 0:\n                        name,value = str(meter.name).upper(),float(meter.avg)\n                        if name in metrics or not math.isfinite(value):\n                            raise ValueError(\'Duplicate/nonfinite native metric.\')\n                        metrics[name] = value\n            for name in (\'RECALL@10\',\'RECALL@20\',\'NDCG@10\',\'NDCG@20\'):\n                if name not in metrics or not 0<=metrics[name]<=1:\n                    raise ValueError(f\'Missing/invalid native metric: {name}\')\n                old = rec[\'historical_selected_test_rounded_4dp\'][name]\n                if abs(metrics[name]-old) > 0.0000501:\n                    raise ValueError(f\'Reevaluation disagrees with historical selected metric precision: {name}\')\n            receipt = dict(method=\'ECLIPSE\',arm=\'wo_ec\',dataset=rec[\'dataset\'],seed=1,mode=\'test\',\n                epoch=epoch,selected_checkpoint=True,selection=\'validation NDCG@20\',metrics=metrics,\n                evidence_mode=\'pooled_activity_excess\',evidence_delta=0.0,\n                graph_fingerprint=rec[\'graph_fingerprint\'],training_reused=True,\n                checkpoint_sha256=sha(checkpoint),checkpoint_path=str(checkpoint),\n                source_hashes=rec[\'source_hashes\'],training_commit=rec[\'training_commit\'],\n                evaluation_runtime=runtime,runtime_deviations=deviations,\n                provenance=\'Original native uncalibrated checkpoint, operator-equivalent to exact delta=0 bypass. \'\n                           \'No native metadata rewritten; TEST evaluator/scorer/seen masking unchanged.\',\n                metrics_precision=\'native AverageMeter averages before step reset\')\n            write(target/\'selected_metrics.json\',receipt)\n            with (target/\'evaluation_metrics.jsonl\').open(\'a\',encoding=\'utf-8\') as f:\n                f.write(json.dumps(receipt,allow_nan=False)+\'\\n\')\n            return result\n    coach = ReceiptCoach(dataset=dataset,model=model,cfg=cfg,\n        trainpipe=model.sure_trainpipe(cfg.batch_size),validpipe=model.sure_validpipe(cfg.ranking),\n        testpipe=model.sure_testpipe(cfg.ranking))\n    started = time.perf_counter()\n    try:\n        coach.test(selected_epoch)  # Native no_grad/test-mode/scoring/masking/monitor path.\n        if not (target/\'selected_metrics.json\').is_file():\n            raise RuntimeError(\'Native TEST receipt not captured.\')\n        write(target/\'run_status.json\',dict(status=\'COMPLETED\',arm=\'wo_ec\',seed=1,\n            training_reused=True,evaluation_only=True,selected_epoch=selected_epoch,\n            reevaluation_seconds=time.perf_counter()-started))\n    finally:\n        coach.shutdown()\n'
helper_path = RUN_ROOT/'helpers/evaluate_wo_ec_original_checkpoint.py'
helper_path.write_text(EVALUATION_HELPER,encoding='utf-8')
import yaml
ARTIFACT_PATHS, LOG_PATHS, PLAN_PATHS, CONFIG_PATHS = {}, {}, {}, {}
for key in SELECTED_DATASETS:
    rec = EXPECTED_AUDIT[key]
    original = rec['legacy_manifest']
    config = yaml.safe_load((CHECKOUTS[key]/'configs'/f'Amazon2014{key.capitalize()}_STAIR5_v8.yaml').read_text(encoding='utf-8'))
    config.update(original['v8_config'])
    config.update(original['checkpoint_contract']['config'])
    artifact = RUN_ROOT/'artifacts'/f'{key}_eclipse_wo_ec_seed1'
    artifact.mkdir()
    config.update(root=str(DATA_ROOT),dataset=DATASET_FOLDERS[key],seed=1,
                  eval_valid=True,eval_test=False,ranking='full',which4best='NDCG@20',
                  resume=False,resume_from='',artifact_dir=str(artifact),
                  graph_cache_dir=str(WORK_DIR/'eclipse_wo_ec_reuse_cache'/rec['training_commit'][:12]/key))
    config['mfiles'] = ','.join(config['mfiles'])
    config['num_neighbors'] = '-'.join(map(str,config['num_neighbors']))
    config['v4_support_files'] = ','.join(config['v4_support_files'])
    config_path = RUN_ROOT/'configs'/f'{key}_eclipse_wo_ec_seed1_reevaluation.yaml'
    config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    for name in ('manifest.json','training_telemetry.jsonl','training.log'):
        shutil.copy2(REUSE_ROOT/key/name,artifact/name)
    shutil.copy2(REUSE_ROOT/key/'best_model.pt',artifact/'best_model.pt')
    plan = dict(output=str(artifact),checkpoint=str(artifact/'best_model.pt'),audit=rec,strict_runtime=STRICT_RUNTIME)
    plan_path = RUN_ROOT/'provenance'/f'{key}_reevaluation_plan.json'
    write_json(plan_path,plan)
    ARTIFACT_PATHS[key],LOG_PATHS[key] = artifact,artifact/'training.log'
    CONFIG_PATHS[key],PLAN_PATHS[key] = config_path,plan_path


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
def read_jsonl(path):
    path=Path(path)
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()] if path.is_file() else []
def reevaluate_dataset(key):
    if key not in SELECTED_DATASETS:
        print('Skipped:',key); return
    artifact=ARTIFACT_PATHS[key]
    status=artifact/'run_status.json'
    if status.is_file() and json.loads(status.read_text())['status']=='COMPLETED':
        print('Reading existing recovery; no reevaluation:',key)
        return json.loads((artifact/'selected_metrics.json').read_text())
    if (artifact/'selected_metrics.json').exists() or (artifact/'evaluation_metrics.jsonl').exists():
        raise FileExistsError('Incomplete recovery preserved; start a new session instead of overwriting.')
    env=dict(ENV,PYTHONPATH=str(CHECKOUTS[key]),ECLIPSE_REUSE_PLAN=str(PLAN_PATHS[key]))
    command=[sys.executable,'-u',str(helper_path),'--config',str(CONFIG_PATHS[key]),'--seed','1',
             '--device','0','--root',str(DATA_ROOT),'--dataset',DATASET_FOLDERS[key]]
    write_json(RUN_ROOT/'provenance'/f'{key}_command.json',dict(argv=command,cwd=str(CHECKOUTS[key]),evaluation_only=True))
    process=None
    try:
        with (RUN_ROOT/'logs'/f'{key}_wo_ec_seed1_reevaluation.log').open('x',encoding='utf-8') as log:
            process=subprocess.Popen(command,cwd=CHECKOUTS[key],env=env,stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush()
            process.wait(); process.stdout.close()
        if process.returncode:
            raise RuntimeError(f'{key}: native checkpoint evaluation failed; inspect the preserved log.')
    finally:
        if process is not None and process.poll() is None:
            process.terminate()
            try: process.wait(timeout=20)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
    if not status.is_file() or json.loads(status.read_text()).get('status')!='COMPLETED':
        raise ValueError('No completed native recovery receipt.')
    receipt=json.loads((artifact/'selected_metrics.json').read_text())
    print('Recovered native selected TEST:',key,receipt['epoch'],receipt['metrics'])
    return receipt


## 5. Figure utilities — PDF vector, PNG 300 dpi, source CSV and captions


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 10,
                     'axes.labelsize': 9, 'legend.fontsize': 8, 'xtick.labelsize': 8,
                     'ytick.labelsize': 8, 'lines.linewidth': 1.5,
                     'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 300,
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'blue': '#0072B2', 'orange': '#D55E00', 'green': '#009E73',
          'purple': '#CC79A7', 'gray': '#666666'}
_existing_figure_manifest = RUN_ROOT / 'figures' / 'figure_manifest.json'
FIGURE_RECORDS = json.loads(_existing_figure_manifest.read_text(encoding='utf-8')) if _existing_figure_manifest.is_file() else []


def save_figure(fig, stem, caption, sources):
    for extension in ('pdf', 'png'):
        fig.savefig(RUN_ROOT / 'figures' / f'{stem}.{extension}', bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    FIGURE_RECORDS[:] = [record for record in FIGURE_RECORDS if record['stem'] != stem]
    FIGURE_RECORDS.append({'stem': stem, 'caption': caption, 'sources': list(map(str, sources))})
    write_json(RUN_ROOT / 'figures' / 'figure_manifest.json', FIGURE_RECORDS)
    print(caption)


def completed_artifact(key):
    if key not in SELECTED_DATASETS:
        print(f'{key}: not selected; no figure generated.')
        return None
    path = ARTIFACT_PATHS[key]
    status = path / 'run_status.json'
    if not status.is_file() or json.loads(status.read_text(encoding='utf-8')).get('status') != 'COMPLETED':
        print(f'{key}: no completed run available; no figure generated.')
        return None
    return path


def plot_learning_curves(key):
    artifact = completed_artifact(key)
    if artifact is None:
        return
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    rows = []
    text = (artifact/'training.log').read_text(encoding='utf-8',errors='replace')
    for line in text.splitlines():
        match = re.search(r'VALID\s+@Epoch:\s*(\d+)\s+>>>',line)
        if match:
            metrics = {name.upper():float(value) for name,value in re.findall(r'((?:RECALL|NDCG)@\d+) Avg:\s*([\d.eE+-]+)',line)}
            rows.append(dict(epoch=int(match.group(1)),**metrics))
    valid = pd.DataFrame(rows).drop_duplicates('epoch',keep='first').sort_values('epoch')
    valid.to_csv(RUN_ROOT/'tables'/f'eclipse_wo_ec_{key}_seed1_validation_log_4dp.csv',index=False)
    selected = EXPECTED_AUDIT[key]['selected_epoch']
    if telemetry.empty or valid.empty:
        print(f'{key}: missing measured curves; no substitute data generated.')
        return
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0), constrained_layout=True)
    for column, label, color in [('loss', 'Total', COLORS['blue']),
                                  ('bpr_loss', 'BPR', COLORS['orange']),
                                  ('weighted_nlgcl_loss', 'Weighted LC', COLORS['green'])]:
        if column in telemetry:
            axes[0].plot(telemetry['epoch'], telemetry[column], label=label, color=color)
    axes[0].set(xlabel='Training epoch', ylabel='Mean training loss', title='Training objective')
    axes[0].legend(frameon=False)
    for ax, metric in zip(axes[1:], ('NDCG@20', 'RECALL@20')):
        if metric in valid:
            ax.plot(valid['epoch'], valid[metric], color=COLORS['blue'], label=f'VALID {metric}')
            point = valid.loc[valid['epoch'] == selected, metric]
            if len(point):
                ax.scatter([selected], [point.iloc[-1]], color=COLORS['orange'], s=22, zorder=3)
        ax.axvline(selected, color=COLORS['gray'], linestyle='--', linewidth=1, label=f'Selected epoch {selected}')
        ax.set(xlabel='Training epoch', ylabel=f'Validation {metric}', title=f'Validation {metric}')
        ax.legend(frameon=False)
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle(f'ECLIPSE w/o EC | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'eclipse_wo_ec_{key}_seed{SEED}_learning_curves',
                'Historical training objective and VALID curves (four-decimal native log precision). Selected epoch comes from the original native best-checkpoint log, not argmax of rounded curves. '
                'TEST is evaluated at the selected checkpoint and is not a learning-curve selection signal.',
                [artifact / 'training_telemetry.jsonl', artifact / 'evaluation_metrics.jsonl', artifact / 'selected_metrics.json'])


def plot_training_memory(key):
    artifact = completed_artifact(key)
    if artifact is None:
        return
    telemetry = pd.DataFrame(read_jsonl(artifact / 'training_telemetry.jsonl'))
    if telemetry.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), constrained_layout=True)
    for column, label, color in [('peak_allocated_gib', 'Peak allocated', COLORS['blue']),
                                  ('peak_reserved_gib', 'Peak reserved', COLORS['orange'])]:
        if column in telemetry:
            axes[0].plot(telemetry['epoch'], telemetry[column] * 1024, label=label, color=color)
    axes[0].set(xlabel='Training epoch', ylabel='CUDA allocator memory (MiB)', title='Training-epoch allocator peaks')
    axes[0].legend(frameon=False)
    axes[1].plot(telemetry['epoch'], telemetry['seconds'], color=COLORS['green'])
    axes[1].set(xlabel='Training epoch', ylabel='Training time (s/epoch)', title='Training time per epoch')
    for ax in axes:
        ax.grid(alpha=.2)
    fig.suptitle(f'ECLIPSE w/o EC | {DATASET_TITLES[key]} | seed {SEED}', fontsize=11)
    save_figure(fig, f'eclipse_wo_ec_{key}_seed{SEED}_training_memory',
                'CUDA allocator peaks are measured in the training subprocess for each training epoch. '
                'They exclude evaluation peaks, other GPU processes and driver memory. No arbitrary hardware ceiling is drawn.',
                [artifact / 'training_telemetry.jsonl'])


def plot_calibration_diagnostics(key):
    artifact=completed_artifact(key)
    if artifact is None:
        return
    row={'dataset':key,'seed':1,'calibration_enabled':False,'evidence_delta':0.0,
         'operator_calibration_dose':0.0,'polynomial_calibration_dose':0.0,
         'dose_provenance':'exact delta-zero operator identity; analytical, not measured Adam-direction dose',
         'evidence_coverage':'NOT_MEASURED','evidence_quantiles':'NOT_MEASURED'}
    table=RUN_ROOT/'tables'/f'eclipse_wo_ec_{key}_seed1_calibration_diagnostics.csv'
    pd.DataFrame([row]).to_csv(table,index=False)
    fig,ax=plt.subplots(figsize=(4.5,2.8),constrained_layout=True)
    ax.bar(['Operator dose','Polynomial dose'],[0,0],color=[COLORS['blue'],COLORS['orange']])
    ax.set(ylim=(0,1),ylabel='Calibration perturbation (%)',title=f'{DATASET_TITLES[key]} | w/o EC | seed 1')
    ax.text(.5,.5,'Calibration disabled: exactly zero dose\nEvidence coverage was not measured',ha='center',va='center',transform=ax.transAxes)
    save_figure(fig,f'eclipse_wo_ec_{key}_seed1_calibration_diagnostics',
        'Operator equivalence establishes zero calibration dose analytically. No legacy evidence statistics '
        'were measured; missing coverage is not replaced by zero. No measured Adam-direction perturbation is claimed.',[table])

def plot_architecture():
    fig, ax = plt.subplots(figsize=(7.2, 2.7))
    ax.axis('off')
    blocks = [('Text / image\nWMSG candidates + weights', COLORS['blue']),
              ('EC BYPASS\nevidence delta = 0', COLORS['orange']),
              ('Semantic / CF blend\neta = 0.1', COLORS['green']),
              ('Item-BSC on Adam direction\nMI/FSC + BPR + LC retained', COLORS['blue'])]
    for i, (label, color) in enumerate(blocks):
        x = .02 + i * .25
        ax.add_patch(FancyBboxPatch((x, .35), .21, .38, boxstyle='round,pad=0.01',
                     edgecolor=color, facecolor=color + '14', transform=ax.transAxes))
        ax.text(x + .105, .54, label, ha='center', va='center', fontsize=8, transform=ax.transAxes)
        if i < 3:
            ax.annotate('', xy=(x+.25,.54), xytext=(x+.21,.54), xycoords='axes fraction',
                        arrowprops={'arrowstyle':'->'})
    ax.set_title('ECLIPSE w/o EC: only calibration is disabled')
    save_figure(fig, 'eclipse_wo_ec_architecture',
        'The ablation keeps semantic weighting, collaborative support expansion, layer-contrastive learning '
        'and optimizer-side BSC. Delta zero routes to the exact uncalibrated operator. '
        'Evidence statistics can still be computed offline, but the calibration operator dose is zero.', [])


In [ ]:
plot_architecture()


## Sports — evaluate existing selected checkpoint, then draw historical curves
No training. No TEST-based reselection.


In [ ]:
receipt = reevaluate_dataset('sports')
if receipt is not None:
    display(pd.DataFrame([receipt['metrics']]))
plot_learning_curves('sports')
plot_training_memory('sports')
plot_calibration_diagnostics('sports')


## Baby — evaluate existing selected checkpoint, then draw historical curves
No training. No TEST-based reselection.


In [ ]:
receipt = reevaluate_dataset('baby')
if receipt is not None:
    display(pd.DataFrame([receipt['metrics']]))
plot_learning_curves('baby')
plot_training_memory('baby')
plot_calibration_diagnostics('baby')


## Electronics — evaluate existing selected checkpoint, then draw historical curves
No training. No TEST-based reselection.


In [ ]:
receipt = reevaluate_dataset('electronics')
if receipt is not None:
    display(pd.DataFrame([receipt['metrics']]))
plot_learning_curves('electronics')
plot_training_memory('electronics')
plot_calibration_diagnostics('electronics')


## 6. Selected TEST, historical training resources and paired seed-1 descriptive comparison

Only native recovered receipts enter the primary results. Original fit time and training memory are historical;
new reevaluation time is reported separately. One matched seed is descriptive evidence, not statistical significance.


In [ ]:
rows=[]
for key in SELECTED_DATASETS:
    artifact=completed_artifact(key)
    if artifact is None:
        rows.append(dict(dataset=key,seed=1,status='NOT_MEASURED')); continue
    receipt=json.loads((artifact/'selected_metrics.json').read_text())
    status=json.loads((artifact/'run_status.json').read_text())
    rec=EXPECTED_AUDIT[key]
    telemetry=pd.DataFrame(read_jsonl(artifact/'training_telemetry.jsonl'))
    telemetry.to_csv(RUN_ROOT/'tables'/f'eclipse_wo_ec_{key}_seed1_training_telemetry.csv',index=False)
    rows.append(dict(method='ECLIPSE',arm='wo_ec',dataset=key,seed=1,status='COMPLETED',
        selected_epoch=receipt['epoch'],split='TEST',**receipt['metrics'],
        historical_fit_seconds=rec['historical_fit_seconds'],
        historical_preprocessing_seconds=rec['legacy_manifest'].get('preprocessing_seconds'),
        historical_peak_training_allocated_mib=float(telemetry['peak_allocated_gib'].max()*1024),
        historical_peak_training_reserved_mib=float(telemetry['peak_reserved_gib'].max()*1024),
        reevaluation_seconds=status['reevaluation_seconds'],training_reused=True,
        graph_fingerprint=rec['graph_fingerprint'],checkpoint_sha256=receipt['checkpoint_sha256'],
        training_commit=rec['training_commit'],runtime_deviations=receipt['runtime_deviations']))
SUMMARY=pd.DataFrame(rows)
table=RUN_ROOT/'tables/eclipse_wo_ec_seed1_selected_test.csv'
SUMMARY.to_csv(table,index=False,na_rep='NOT_MEASURED')
display(SUMMARY.fillna('NOT_MEASURED'))
measured=SUMMARY.loc[SUMMARY['status']=='COMPLETED'].copy()
if len(measured):
    labels=[DATASET_TITLES[k].replace('Amazon ','') for k in measured['dataset']]
    fig,axes=plt.subplots(1,3,figsize=(7.2,2.9),constrained_layout=True)
    x=np.arange(len(measured))
    axes[0].bar(x,measured['historical_fit_seconds']/3600,color=COLORS['blue'])
    axes[0].set(ylabel='Historical Coach.fit time (h)',title='Original training + native I/O')
    axes[1].bar(x,measured['historical_peak_training_allocated_mib'],color=COLORS['green'])
    axes[1].set(ylabel='Historical training peak (MiB)',title='Allocated; evaluation excluded')
    axes[2].bar(x,measured['reevaluation_seconds'],color=COLORS['orange'])
    axes[2].set(ylabel='New native TEST evaluation (s)',title='Checkpoint recovery only')
    for ax in axes: ax.set_xticks(x,labels); ax.grid(axis='y',alpha=.2)
    save_figure(fig,'eclipse_wo_ec_seed1_resource_profile',
                'Historical training resources and new checkpoint reevaluation are separate measurements. '
                'Different hardware/session dates must not be treated as controlled speed comparisons.',[table])
    paired=[]
    for _,row in measured.iterrows():
        for metric in PRIMARY_METRICS:
            full=SEED1_REFERENCE[row['dataset']]['selected_receipt']['metrics'][metric]
            base=float(row[metric])
            paired.append(dict(dataset=row['dataset'],seed=1,metric=metric,full=full,wo_ec=base,
                               absolute_difference=full-base,relative_difference_percent=100*(full-base)/base))
    PAIRED=pd.DataFrame(paired)
    pair_table=RUN_ROOT/'tables/eclipse_ec_seed1_paired_descriptive.csv'
    PAIRED.to_csv(pair_table,index=False)
    fig,axes=plt.subplots(1,2,figsize=(7.2,2.8),constrained_layout=True)
    for ax,metric in zip(axes,('NDCG@20','RECALL@20')):
        subset=PAIRED.loc[PAIRED['metric']==metric]
        ax.axhline(0,color=COLORS['gray'],lw=1)
        ax.bar([DATASET_TITLES[k].replace('Amazon ','') for k in subset['dataset']],
               subset['absolute_difference'],color=COLORS['blue'])
        ax.set(ylabel=f'Full - w/o EC: {metric}',title='One matched seed; descriptive only')
        ax.grid(axis='y',alpha=.2)
    save_figure(fig,'eclipse_ec_seed1_paired_descriptive',
        'Absolute differences between full ECLIPSE and w/o EC at their own validation-selected checkpoints. '
        'One training seed per dataset; no error bars or p-values. This is not a multi-seed ablation result.',[pair_table])
write_json(RUN_ROOT/'tables/statistical_analysis_status.json',
    dict(n_training_seeds=1,p_value='NOT_COMPUTED',sample_std='NOT_MEASURED',
         reason='Recovered seed 1 is the same original training attempt, not an additional independent seed.'))


## 7. Archive and next jobs

Download this output ZIP and pass its artifacts to the w/o EC seed-2/3 aggregation notebook.
Only after successful native recovery should the plan's three seed-1 jobs become `REUSED_VERIFIED`.
If a strict contract/hash/runtime check fails, preserve the error and resolve it before reporting results.
No checkpoint metadata bypass and no automatic 500-epoch fallback.


In [ ]:
entries=json.loads((RUN_ROOT/'figures/figure_manifest.json').read_text()) if (RUN_ROOT/'figures/figure_manifest.json').is_file() else []
captions='# ECLIPSE w/o EC seed 1 — figure captions\n\n'
for entry in entries:
    captions+=f"## {entry['stem']}\n\n{entry['caption']}\n\nSources: "+', '.join(entry['sources'])+'\n\n'
(RUN_ROOT/'figures/FIGURE_CAPTIONS.md').write_text(captions,encoding='utf-8')
write_json(RUN_ROOT/'run_manifest.json',dict(method='ECLIPSE',arm='wo_ec',seed=1,training_reused=True,
    evaluation_only=True,runtime=RUNTIME_ENV,runtime_deviations=deviations,
    datasets=list(SELECTED_DATASETS),selection='validation NDCG@20',
    original_training_commits={k:EXPECTED_AUDIT[k]['training_commit'] for k in SELECTED_DATASETS}))
archive=WORK_DIR/f'{RUN_ROOT.name}.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file() and not path.is_relative_to(RUN_ROOT/'verified_reuse_input'):
            z.write(path,str(Path(RUN_ROOT.name)/path.relative_to(RUN_ROOT)))
print('Archive:',archive,'MiB:',archive.stat().st_size/2**20,'SHA256:',file_sha256(archive))
from IPython.display import FileLink
display(FileLink(str(archive)))
